# Folio Python 演示 — v0.2.1

对应 `python-demo.py` 的 Jupyter 版本。每个 cell 单独运行，建议录制时逐格展示。

**前置条件（一次性）：**
```bash
pip install folio-docx
```

## 1. 安装与版本验证

In [ ]:
# 安装一次（在 notebook 的第一个 cell 里运行；之后注释掉）
# !pip install folio-docx

import folio
import os

print('folio version:', folio.__version__)
print('built-in themes:', folio.list_themes())

## 2. Markdown 字符串 → .docx 字节（默认样式）

In [ ]:
markdown = """# Hello, Folio

Inline math: $E = mc^2$

Display math:

$$\\int_{0}^{1} x^2 \\, dx = \\frac{1}{3}$$

| Theme | Use case |
|---|---|
| academic | English papers |
| thesis-cn | 中文论文 |
| report | Business memos |
"""

data = folio.convert(markdown)
print(f'output is {len(data):,} bytes; DOCX magic = {data[:2]!r}')

os.makedirs('demo/outputs', exist_ok=True)
with open('demo/outputs/nb-default.docx', 'wb') as f:
    f.write(data)
print('→ wrote demo/outputs/nb-default.docx')

## 3. 内置主题（academic / thesis-cn / report）

In [ ]:
# 英文学术：Times New Roman 12pt + 1.5 倍行距
d = folio.convert(markdown, theme='academic')
with open('demo/outputs/nb-academic.docx', 'wb') as f: f.write(d)
print(f'academic  → {len(d):,} bytes')

# 中文论文：宋体正文 + 黑体标题 + 首行缩进 2 字符
d = folio.convert(markdown, theme='thesis-cn')
with open('demo/outputs/nb-thesis-cn.docx', 'wb') as f: f.write(d)
print(f'thesis-cn → {len(d):,} bytes')

# 商务报告：Calibri + 蓝色点缀
d = folio.convert(markdown, theme='report')
with open('demo/outputs/nb-report.docx', 'wb') as f: f.write(d)
print(f'report    → {len(d):,} bytes')

## 4. 中文内容 + thesis-cn 主题

In [ ]:
cn_md = """# 引言

本论文研究 Markdown 到 Word 转换的工程实践。

行内公式：当 $a \\ne 0$，一元二次方程有以下解。

## 公式

$$x = \\frac{-b \\pm \\sqrt{b^2 - 4ac}}{2a}$$

## 表格

| 章节 | 内容 |
|---|---|
| 第一章 | 引言 |
| 第二章 | 方法 |
| 第三章 | 实验结果 |
"""

d = folio.convert(cn_md, theme='thesis-cn')
with open('demo/outputs/nb-thesis-cn-custom.docx', 'wb') as f: f.write(d)
print(f'→ demo/outputs/nb-thesis-cn-custom.docx  ({len(d):,} bytes)')

## 5. 文件到文件转换（图片路径相对于 input 文件目录解析）

In [ ]:
# 英文 demo（含 SVG + PNG 图片）→ academic 主题
folio.convert_file(
    'demo/demo-en.md',
    'demo/outputs/nb-file-academic.docx',
    theme='academic',
)
print('→ demo/outputs/nb-file-academic.docx')

# 中文 demo（含图片）→ thesis-cn 主题
folio.convert_file(
    'demo/demo-cn.md',
    'demo/outputs/nb-file-thesis-cn.docx',
    theme='thesis-cn',
)
print('→ demo/outputs/nb-file-thesis-cn.docx')

## 6. 顶会 / 期刊模板（reference-doc）

templates 目录里已准备了 ACM、IEEE、Springer LNCS 三份模板。

In [ ]:
templates = sorted(
    os.path.join('demo/templates', f)
    for f in os.listdir('demo/templates')
    if f.endswith('.docx')
)
print('available templates:', templates)

In [ ]:
# 用每个模板转换 demo-en.md
for tmpl in templates:
    name = os.path.splitext(os.path.basename(tmpl))[0]
    out = f'demo/outputs/nb-ref-{name}.docx'
    folio.convert_file('demo/demo-en.md', out, reference_doc=tmpl)
    size = os.path.getsize(out)
    print(f'  {name:<20} → {out}  ({size:,} bytes)')

## 7. 互斥保护：theme 和 reference_doc 不能同时用

In [ ]:
try:
    folio.convert('# test', theme='academic', reference_doc=templates[0])
except ValueError as e:
    print(f'expected ValueError: {e}')

## 8. HTML 预览（桌面端使用同一份渲染器）

In [ ]:
from IPython.display import HTML

# preview_html 返回不带 <html> 外壳的片段
HTML(folio.preview_html(
    '# Folio Preview\n\n**bold** and $E = mc^2$.\n\n```rust\nfn main() {}\n```'
))

## 9. 多线程批量转换（convert() 主动释放 GIL）

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

batch = [f'# Document {i}\n\nBody text, $x_{{{i}}} = {i}^2$.\n' for i in range(20)]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    outs = list(pool.map(folio.convert, batch))
elapsed = time.perf_counter() - start

print(f'converted {len(outs)} docs in {elapsed:.3f}s (8 threads)')
print(f'all valid .docx: {all(o[:2] == b"PK" for o in outs)}')

## 10. 输出文件汇总

In [ ]:
print('demo/outputs/ 目录下的文件：')
for fname in sorted(os.listdir('demo/outputs')):
    full = os.path.join('demo/outputs', fname)
    size = os.path.getsize(full)
    print(f'  {fname:<45}  {size:>9,} bytes')